# 🧬 ⚙️ CpGPT Synthetic Data Generation Tutorial ⚙️ 🧬

Welcome to the CpGPT Synthetic Data Generation Tutorial! 👋 

In this notebook, we'll walk you through the process of generating synthetic DNA methylation data from a pre-trained CpGPT model.

## Table of Contents

1. [Setup Environment](#1-setup-environment)
2. [Load CpGPT Dependencies](#2-load-cpgpt-dependencies)
3. [Load Pre-trained Model](#3-load-pre-trained-model)
4. [Generate DNA LLM Embeddings](#4-generate-dna-llm-embeddings)
5. [Synthetic Data Generation](#5-synthetic-data-generation)
6. [Visualize Synthetic Methylome](#6-visualize-synthetic-methylome)
7. [Next Steps](#7-next-steps)

## 1. Setup Environment

We'll import the necessary Python packages and set up our environment for CpGPT. We'll be using a mix of standard data science libraries and CpGPT-specific modules. We'll also set some important variables that will be used throughout the notebook. Pay attention to these as you may need to adjust them based on your specific setup and requirements. The bulk of the variables are defined here below. The pre-trained model checkpoint alongside DNA embedding files can also be downloaded if not present.

CpGPT model files and DNA-sequence dependencies are hosted on Hugging Face.
The download cell below downloads any missing files into the standard Hugging Face cache and reuses them on later runs; no command-line download is needed.

### 1.1 Download dependencies and define variables

In [ ]:
from cpgpt import download_cpgpt

resources = download_cpgpt(model="small", species="human")

In [ ]:
# Random seed for reproducibility
RANDOM_SEED = 42

# Directory paths
ARROW_DF_PATH = "../data/tutorials/raw/toy.arrow"

# Device configuration
DEVICE = "cuda"

LLM_DEPENDENCIES_DIR = str(resources.dependencies_path)
DEPENDENCIES_DIR = str(resources.dependencies_path)
MODEL_CHECKPOINT_PATH = str(resources.checkpoint_path)
MODEL_CONFIG_PATH = str(resources.config_path)
MODEL_VOCAB_PATH = str(resources.vocab_path) if resources.vocab_path is not None else None

### 1.2 Import packages


In [ ]:
# Standard library imports
import warnings

warnings.simplefilter(action="ignore", category=FutureWarning)

import matplotlib as mpl
import numpy as np
import pandas as pd
import plotly.express as px

# Plotting imports
import plotly.graph_objects as go
import pyaging as pya
import torch

# Lightning imports
from lightning.pytorch import seed_everything
from plotly.subplots import make_subplots
from scipy.stats import gaussian_kde
from sklearn.decomposition import PCA

# cpgpt-specific imports
from cpgpt.data.components.dna_llm_embedder import DNALLMEmbedder
from cpgpt.data.components.illumina_methylation_prober import IlluminaMethylationProber
from cpgpt.infer.cpgpt_inferencer import CpGPTInferencer

# Set random seed for reproducibility
seed_everything(RANDOM_SEED, workers=True)

## 2. Load CpGPT Dependencies

Now that we have our basic environment set up, we need to load three crucial components for CpGPT:

1. **DNA LLM Embedder**: This component contains the embeddings for all CpGs of interest. It's essential for converting DNA sequences into a format that our model can understand and process.

2. **Illumina Methylation Prober**: This tool allows us to convert probe names to genomic locations, which is required for working with Illumina methylation array data.

3. **CpG Inferencer**: This component is responsible for loading the pre-trained CpGPT model checkpoint and any additional configurations or overrides needed for the fine-tuning process.

### 3.1 Load DNALLMEmbedder

In [ ]:
embedder = DNALLMEmbedder(dependencies_dir=DEPENDENCIES_DIR)

### 3.2 Load IlluminaMethylationProber

In [ ]:
prober = IlluminaMethylationProber(dependencies_dir=DEPENDENCIES_DIR, embedder=embedder)

### 3.3 Load CpGInferencer

In [ ]:
inferencer = CpGPTInferencer()

## 3. Load Pre-trained Model

We need to load a pre-trained CpGPT model to be able to reconstruct the beta values for different CpG sites. Generally, the pre-trained model works well for unconditional generation but you might want to finetune the model using data that has a label for conditional generation.

In [ ]:
config = inferencer.load_cpgpt_config(MODEL_CONFIG_PATH)
model_info = {"current_hparams": config}
model = inferencer.load_cpgpt_model(
    config,
    model_ckpt_path=MODEL_CHECKPOINT_PATH,
    strict_load=True,
)

## 4. Generate DNA LLM Embeddings
 
In this section, we'll generate DNA embeddings for the all of the probes in any Illumina methylation array that might be missing. The model requires that all genomic locations have embeddings from a DNA LLM or it will fail otherwise. The easiest way is to download such embeddings if you are working with human samples ([step 1.3](#1.3-download-dependencies)) or it might take a little bit for the processing to happen.

In [ ]:
# Get all Illumina probes
probes = list(prober.illumina_metadata_dict["homo_sapiens"].keys())[:60000]

# Retrieve genomic location of all Illumina probes
genomic_locations = list(prober.illumina_metadata_dict["homo_sapiens"].values())[:60000]

# Embed all Illumina probes
embedder.parse_dna_embeddings(
    genomic_locations,
    species="homo_sapiens",
    dna_llm=model_info["current_hparams"]["data"]["dna_llm"],
    dna_context_len=model_info["current_hparams"]["data"]["dna_context_len"],
)

## 5. Synthetic Data Generation
 
In order to generate synthetic DNA methylation data, we need to choose between two approaches:
 
1. Unconditional Generation:
    - CpGPT generates samples from anything it has been trained on.
    - This approach is useful for unsupervised learning if you have a finetuned model.
    - It doesn't require a specific label for generation.
 
2. Conditional Generation:
    - A finetuned CpGPT model predicts a specific label to guide the generation process.
    - This approach is useful for generating synthetic data similar to a specific real dataset.
    - It requires a model trained on labeled data.
 
The choice between these approaches depends on your specific use case and the availability of labeled data.

### 5.1 Unconditional Generation

In [ ]:
# Generate embeddings
sample_embedding_uncond = inferencer.generate_sample_embedding(model, num_samples=4, batch_size=16)

In [ ]:
# Reconstruct beta values
X_uncond, X_uncond_unc = inferencer.reconstruct_betas(
    model,
    sample_embedding=sample_embedding_uncond,
    genomic_locations=genomic_locations,
    embedder=embedder,
    dna_llm=model_info["current_hparams"]["data"]["dna_llm"],
    dna_context_len=model_info["current_hparams"]["data"]["dna_context_len"],
    species="homo_sapiens",  # You need to pick a single species
    batch_size=1,
)

In [ ]:
# Transform to DataFrame
X_uncond_df = pd.DataFrame(X_uncond.detach().cpu().numpy(), columns=probes)
X_uncond_unc_df = pd.DataFrame(X_uncond_unc.detach().cpu().numpy(), columns=probes)

In [ ]:
# Transform to DataFrame
X_cond_df = X_uncond_df
X_cond_unc_df = X_uncond_unc_df

### 5.2 Conditional Generation

> **⚠️ Warning**
> 
> You need a finetuned model that has been trained on the same label you want to condition on.
> 
> Otherwise, the conditional generation will not work.

In [ ]:
# Generate embeddings
sample_embedding_cond = inferencer.generate_sample_embedding(
    model,
    num_samples=100,
    obsm=torch.tensor([5.0]).to(DEVICE),
    guidance_scale=0.5,
    batch_size=16,
)

In [ ]:
# Reconstruct beta values
X_cond, X_cond_unc = inferencer.reconstruct_betas(
    model,
    sample_embedding=sample_embedding_cond,
    genomic_locations=genomic_locations,
    embedder=embedder,
    dna_llm=model_info["current_hparams"]["data"]["dna_llm"],
    dna_context_len=model_info["current_hparams"]["data"]["dna_context_len"],
    species="homo_sapiens",  # You need to pick a single species
    batch_size=1,
)

In [ ]:
# Transform to DataFrame
X_cond_df = pd.DataFrame(X_cond.detach().cpu().numpy(), columns=probes)
X_cond_unc_df = pd.DataFrame(X_cond_unc.detach().cpu().numpy(), columns=probes)

## 6. Visualize Synthetic Methylome

As a quick sanity check, let's create some plots to try to understand what is going on. We'll visualize:

1. The distribution of beta values for the synthetic samples vs real data
2. The PCA plot of real samples vs unconditional, and conditional synthetic samples

These visualizations will help us ensure that our model is working as expected. If not, it might be worth finetuning with the appropriate data.

In [ ]:
# Load the real data
df = pd.read_feather(ARROW_DF_PATH)

# Identify common columns
common_columns = set(df.columns) & set(X_uncond_df.columns) & set(X_cond_df.columns)
common_columns = list(common_columns - {"type"})  # Exclude 'type' if present

### 6.1 Distribution of beta values

In [ ]:
# Create three panels of histograms side by side for real, unconditional, and conditional data

# Use only common columns and the same number of samples for each type
subset_size = min(len(df), len(X_uncond_df), len(X_cond_df))
subset_size = 1  # int(0.5 * subset_size)  # Use 50% of the smallest dataset

# Function to create a subset dataframe


def create_subset(dataframe, size):
    return (
        dataframe[common_columns]
        .sample(n=size, random_state=42)
        .melt(var_name="CpG Site", value_name="Beta Value")
    )


# Create subsets
real_subset = create_subset(df, subset_size)
uncond_subset = create_subset(X_uncond_df, subset_size)
cond_subset = create_subset(X_cond_df, subset_size)

# Function to calculate histogram and KDE data


def calc_hist_kde(data):
    hist, bin_edges = np.histogram(data, bins=50, range=(0, 1), density=True)
    bin_centers = (bin_edges[:-1] + bin_edges[1:]) / 2
    kde = gaussian_kde(data)
    x_range = np.linspace(0, 1, 200)
    y_kde = kde(x_range)
    return hist, bin_centers, x_range, y_kde


# Calculate histogram and KDE data for each subset
real_hist, real_bins, real_x, real_kde = calc_hist_kde(real_subset["Beta Value"])
uncond_hist, uncond_bins, uncond_x, uncond_kde = calc_hist_kde(uncond_subset["Beta Value"])
cond_hist, cond_bins, cond_x, cond_kde = calc_hist_kde(cond_subset["Beta Value"])

# Create the interactive plot with three subplots
fig = make_subplots(
    rows=1,
    cols=3,
    subplot_titles=("Real Data", "Unconditional Synthetic", "Conditional Synthetic"),
    shared_yaxes=True,
)

# Function to add traces to a subplot


def add_subplot(row, col, hist, bins, x, kde, color, data) -> None:
    fig.add_trace(
        go.Bar(x=bins, y=hist, name="Histogram", marker_color=color, opacity=0.7),
        row=row,
        col=col,
    )
    fig.add_trace(
        go.Scatter(x=x, y=kde, name="KDE", line={"color": darken_color(color)}),
        row=row,
        col=col,
    )
    mean = np.mean(data)
    median = np.median(data)
    fig.add_vline(
        x=mean,
        line_dash="dash",
        line_color="red",
        annotation_text=f"Mean: {mean:.3f}",
        annotation_position="bottom right",
        row=row,
        col=col,
    )
    fig.add_vline(
        x=median,
        line_dash="dash",
        line_color="green",
        annotation_text=f"Median: {median:.3f}",
        annotation_position="top left",
        row=row,
        col=col,
    )


# Function to darken a color
def darken_color(color):
    rgb = mpl.colors.to_rgb(color)
    darker_rgb = tuple(max(0, c - 0.2) for c in rgb)
    return mpl.colors.to_hex(darker_rgb)


# Add subplots
add_subplot(1, 1, real_hist, real_bins, real_x, real_kde, "skyblue", real_subset["Beta Value"])
add_subplot(
    1,
    2,
    uncond_hist,
    uncond_bins,
    uncond_x,
    uncond_kde,
    "lightcoral",
    uncond_subset["Beta Value"],
)
add_subplot(1, 3, cond_hist, cond_bins, cond_x, cond_kde, "lightgreen", cond_subset["Beta Value"])

# Update layout
fig.update_layout(
    title=f"Distribution of Beta Values: Real vs Synthetic ({subset_size} samples each)",
    xaxis_title="Beta Value",
    yaxis_title="Density",
    legend_title="",
    hovermode="x unified",
    width=1200,
    height=400,
)

# fig.update_xaxes(range=[0, 1])
fig.update_yaxes(title_text="Density", row=1, col=1)

# Show the plot
fig.show()

### 6.2 PCA of real vs synthetic methylomes

In [ ]:
# Combine real and synthetic data using only common columns
combined_data = pd.concat(
    [
        df[common_columns].assign(type="Real"),
        X_uncond_df[common_columns].assign(type="Unconditional Synthetic"),
        X_cond_df[common_columns].assign(type="Conditional Synthetic"),
    ],
    ignore_index=True,
)

# Perform PCA
pca = PCA(n_components=2)
pca_result = pca.fit_transform(combined_data[common_columns])

# Create a DataFrame with PCA results
pca_df = pd.DataFrame(data=pca_result, columns=["PC1", "PC2"])
pca_df["type"] = combined_data["type"].values

# Create the scatter plot
fig = px.scatter(
    pca_df,
    x="PC1",
    y="PC2",
    color="type",
    title="PCA of Real vs Synthetic Methylomes",
    labels={"PC1": "Principal Component 1", "PC2": "Principal Component 2"},
    color_discrete_map={
        "Real": "blue",
        "Unconditional Synthetic": "red",
        "Conditional Synthetic": "green",
    },
)

# Update layout
fig.update_layout(legend_title="Data Type", width=800, height=600)

# Show the plot
fig.show()

### 6.3 Epigenetic clock prediction

In [ ]:
# List of clocks
clocks = ["horvath2013", "skinandblood", "altumage"]

# Prepare AnnData objects for real, unconditional, and conditional data
adata_real = pya.pp.df_to_adata(df, imputer_strategy="mean", verbose=False)
adata_uncond = pya.pp.df_to_adata(X_uncond_df, imputer_strategy="mean", verbose=False)
adata_cond = pya.pp.df_to_adata(X_cond_df, imputer_strategy="mean", verbose=False)

# Predict age using different clocks for each dataset
for adata, data_type in [
    (adata_real, "Real"),
    (adata_uncond, "Unconditional"),
    (adata_cond, "Conditional"),
]:
    pya.pred.predict_age(adata, clocks, verbose=False)

# Compare predictions
comparison_df = pd.concat(
    [
        adata_real.obs[clocks].assign(type="Real"),
        adata_uncond.obs[clocks].assign(type="Unconditional"),
        adata_cond.obs[clocks].assign(type="Conditional"),
    ],
)

# Create subplots
fig = make_subplots(rows=1, cols=3, subplot_titles=clocks)

# Colors for each data type
colors = {"Real": "blue", "Unconditional": "red", "Conditional": "green"}

# Calculate the number of bins
n_bins = 30  # You can adjust this number as needed

# Visualize comparisons with overlapping histograms
for i, clock in enumerate(clocks):
    # Calculate the range for all data types
    all_data = comparison_df[clock]
    min_val, max_val = all_data.min(), all_data.max()

    for data_type in ["Real", "Unconditional", "Conditional"]:
        data = comparison_df[comparison_df["type"] == data_type][clock]
        fig.add_trace(
            go.Histogram(
                x=data,
                name=data_type,
                opacity=0.7,
                marker_color=colors[data_type],
                histnorm="percent",  # Show proportions instead of counts
                nbinsx=n_bins,  # Set the number of bins
                xbins={
                    "start": min_val,
                    "end": max_val,
                    "size": (max_val - min_val) / n_bins,
                },  # Set the bin range
            ),
            row=1,
            col=i + 1,
        )

    fig.update_xaxes(title_text="Predicted Age", row=1, col=i + 1)
    fig.update_yaxes(title_text="Percentage", row=1, col=i + 1)

# Update layout
fig.update_layout(
    height=400,
    width=1200,
    title_text="Epigenetic Clock Predictions: Real vs Synthetic Data",
    barmode="overlay",
)

# Show the plot
fig.show()

## 7. Next steps

Here are some ideas for further exploration:

- Experiment with different strengths for the guidance scale for conditional generation
- Use various pre-trained CpGPT models and compare they compare to real data
- Predict age with other epigenetic clocks and compare the results